# Reweighting Analysis for cpAEDS Simulations

This notebook demonstrates the exponential reweighting workflow for constant-pH
AEDS simulations. Given a single simulation run at a fixed EIR offset, the
reweighting approach reconstructs the protonation fraction at **arbitrary nearby
offset values** — eliminating the need for a full titration series.

## Workflow
1. Run one (or a few) cpAEDS simulations at fixed EIR offsets
2. Post-process to produce `e*.dat`, `eds_vr.dat`, `eds_vmix.dat`, `df.out`
3. Use `parallel_reweight` to scan over artificial EIR values
4. Plot titration curves with `ReweightPlot`

## Method
$$\langle Q \rangle_{\text{rew}} = \frac{\langle Q \cdot e^{-\beta(H_i - H_{\text{ref}})} \rangle}{\langle e^{-\beta(H_i - H_{\text{ref}})} \rangle}$$

where $Q$ is a binary state indicator (1 = protonated, 0 = deprotonated),
$H_i$ is the EDS reference energy at artificial offset $i$, and
$H_{\text{ref}}$ is the simulation reference energy from `eds_vr.dat`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from cpaeds.reweighting import reweighting_constpH, parallel_reweight, ReweightResult
from cpaeds.reweighting_plots import ReweightPlot

## 1. Demonstrate with synthetic data

The test fixture in `cpaeds/tests/test_data/reweighting/` contains small synthetic
energy files (5 frames, 2 states) that can be used to verify the workflow.

In [ ]:
# Path to synthetic fixture data (included with the package for testing)
FIXTURE_DIR = str(Path("../cpaeds/tests/test_data/reweighting").resolve())

# Run a single reweighting calculation
result = reweighting_constpH(
    cutoff=-400.0,        # energy cutoff (kJ/mol): frames below this go to group A
    temp=300.0,           # temperature (K)
    stepsize=0.5,         # MD output time step (ps)
    itime=0.0,            # starting time (ps)
    group=[["1"], ["0"]], # state groups: [[protonated states], [deprotonated states]]
    path=FIXTURE_DIR,
    H1_eds=False,         # use raw eds_vmix.dat
)

print(f"A_frac  = {result.A_frac:.4f}  (protonated fraction)")
print(f"B_frac  = {result.B_frac:.4f}  (deprotonated fraction)")
print(f"A_frac + B_frac = {result.A_frac + result.B_frac:.6f}  (should be 1.0)")
print(f"emin = {result.emin:.1f} kJ/mol")
print(f"emax = {result.emax:.1f} kJ/mol")

## 2. Scan over artificial EIR values (single run)

By varying `art_eir` we can estimate what the protonation fraction *would have been*
had the simulation been run at a different offset. This is the core idea of reweighting.

In [ ]:
# Scan -250 to -190 kJ/mol for the first state offset
eir_scan = np.linspace(-250.0, -190.0, 61)

scan_results = []
for eir in eir_scan:
    r = reweighting_constpH(
        cutoff=-400.0,
        temp=300.0,
        stepsize=0.5,
        itime=0.0,
        group=[["1"], ["0"]],
        path=FIXTURE_DIR,
        H1_eds=True,
        art_eir=[eir, 0.0],  # vary state-0 offset; state-1 held at 0
    )
    scan_results.append(r)

A_fracs = np.array([r.A_frac for r in scan_results])

fig, ax = plt.subplots(dpi=150)
ax.plot(eir_scan, A_fracs, 'o-', ms=3)
ax.set_xlabel("Artificial EIR (kJ/mol)")
ax.set_ylabel("Reweighted A fraction")
ax.set_title("Single-run reweighted protonation fraction")
plt.tight_layout()
plt.show()

## 3. Multi-run parallel reweighting

In practice you will have multiple independent simulation runs (statistical repeats).
`parallel_reweight` processes all of them in parallel and returns a
`[n_runs][n_eir]` nested list of `ReweightResult` objects.

For a real dataset, replace `path_template` with the path to your `ene_ana` directories:
```python
results = parallel_reweight(
    runs=[1, 2, 3, 4],
    lnspace=np.linspace(-250.0, -190.0, 61),
    nthreads=8,
    path_template="/data/ASP/ASPD_trans_aq_{run}/ene_ana",
    cutoff=-400.0,
    temp=300.0,
    stepsize=0.5,
    itime=4000.0,
    group=[["1"], ["0"]],
    H1_eds=True,
)
```

Below we simulate multiple runs by wrapping the synthetic fixture multiple times.

In [ ]:
# Simulate 3 runs using the same fixture (in practice these would be different paths)
# We add small random noise to the fractions to mimic run-to-run variability
rng = np.random.default_rng(42)

n_runs = 3
simulated_results = []
for _ in range(n_runs):
    run_results = []
    for eir in eir_scan:
        r = reweighting_constpH(
            cutoff=-400.0,
            temp=300.0,
            stepsize=0.5,
            itime=0.0,
            group=[["1"], ["0"]],
            path=FIXTURE_DIR,
            H1_eds=True,
            art_eir=[eir, 0.0],
        )
        # Perturb slightly to simulate different runs
        noise = float(rng.normal(0, 0.02))
        run_results.append(ReweightResult(
            mix=r.mix, reference=r.reference,
            A_rew=r.A_rew, B_rew=r.B_rew,
            A_frac=float(np.clip(r.A_frac + noise, 0.01, 0.99)),
            B_frac=float(np.clip(r.B_frac - noise, 0.01, 0.99)),
            p_A=r.p_A, p_B=r.p_B,
            emin=r.emin, emax=r.emax,
        ))
    simulated_results.append(run_results)

print(f"Collected {len(simulated_results)} runs × {len(simulated_results[0])} EIR points")

## 4. Visualise with ReweightPlot

In [ ]:
rp = ReweightPlot(
    results=simulated_results,
    lnspace=eir_scan,
    ref_eir=-220.0,  # the actual EIR offset used in the simulation
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=150)

# Left: reweighted fraction vs ΔOffset
rp.plot_A_rew(ax=axes[0])
axes[0].set_title("Reweighted protonation fraction")

# Right: reconstructed titration curve
PKA = 4.0  # replace with experimental pKa for your system
rp.plot_titration(pKa=PKA, ax=axes[1])
axes[1].set_title("Reconstructed titration curve")

plt.tight_layout()
plt.show()

## 5. Logistic fit

Fit the mean reweighted fraction to a logistic curve to extract the pKa
(inflection point = pKa in the offset domain, convertible via the
`offset → pH` mapping).

In [ ]:
fig, ax = plt.subplots(dpi=150)
rp.plot_A_rew(ax=ax, show_mean=True, show_band=True)
ax, popt = rp.plot_logfit(ax=ax, color='red')

print(f"Logistic fit parameters: a={popt[0]:.3f}, b={popt[1]:.3f}, c={popt[2]:.3f}, d={popt[3]:.3f}")
print(f"Midpoint (inflection) at ΔOffset = {popt[3]:.1f} kJ/mol")

ax.set_title("Mean A_rew with logistic fit")
plt.tight_layout()
plt.show()

## 6. Accessing raw results

Each `ReweightResult` is a `NamedTuple` — all fields are directly accessible.

In [ ]:
# Access the result for run 0 at EIR scan point 30
r = simulated_results[0][30]

print("ReweightResult fields:")
for field in ReweightResult._fields:
    val = getattr(r, field)
    if hasattr(val, '__len__') and len(val) > 4:
        print(f"  {field:12s}: array of length {len(val)}")
    else:
        print(f"  {field:12s}: {val}")

## Notes

- **Cutoff**: The `-400 kJ/mol` energy cutoff classifies frames into "protonated" (group A)
  and "deprotonated" (group B). Adjust based on your system.
- **H1_eds vs H1_aeds**: `H1_eds=True` uses the raw EDS log-sum-exp as the reference
  Hamiltonian. `H1_aeds=True` adds a soft-core quadratic correction in the `[emin, emax]`
  transition region for better overlap.
- **art_eir**: The artificial per-state EIR offsets. Only the first element (state 0)
  is scanned; the rest are typically held at 0.
- **Parallel runs**: Use `parallel_reweight` for production — it parallelises over EIR
  scan points using `multiprocessing.Pool`.